In [19]:
import pandas as pd
import re
import nltk
import os
import sys
sys.path.append(os.path.abspath(".."))

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
from src.datapreprocessing import load_data,preprocess_text


# Thematic Analysis

## Download NLTK Stopwords

In [13]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\bemnet\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## Load Sentiment-enhanced Review Dataset.

In [14]:
df = load_data("../data/processed/bank_reviews_sentiment.csv")
df.head()

,review_id,review,rating,date,bank,source,sentiment,confidence_score
0,06f6640c-b65c-43e4-88ef-0a79be8b9534,it's a good application,5,2026-05-13,Commercial Bank of Ethiopia,Google Play,Positive,0.999866
1,eda74236-f7f3-4422-a139-a181e832bc27,thank you cbe,5,2026-05-13,Commercial Bank of Ethiopia,Google Play,Positive,0.999756
2,ff53332b-2e76-46d6-83d3-f93f968a4b18,is good,5,2026-05-13,Commercial Bank of Ethiopia,Google Play,Positive,0.999839
3,363a5616-ed3d-4274-85ee-77071067f81d,wow,5,2026-05-13,Commercial Bank of Ethiopia,Google Play,Positive,0.999592
4,56185597-d29b-4a60-a0fb-6783638230a7,Good application,2,2026-05-13,Commercial Bank of Ethiopia,Google Play,Positive,0.999855


## Text Preprocessing

In [15]:

df["clean_review"] = df["review"].apply(preprocess_text)

print(df[["review", "clean_review"]].head())

                    review      clean_review
0  it's a good application  good application
1            thank you cbe         thank cbe
2                  is good              good
3                      wow               wow
4         Good application  good application


## Extract Keywords Using TF-IDF

In [16]:
vectorizer = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 2),   # unigrams + bigrams
    max_features=100
)

X = vectorizer.fit_transform(df["clean_review"])

In [18]:
keywords = vectorizer.get_feature_names_out()

scores = X.mean(axis=0).A1

tfidf_df = pd.DataFrame({
    "keyword": keywords,
    "score": scores
})

tfidf_df = tfidf_df.sort_values(
    by="score",
    ascending=False
)

print("\nTop TF-IDF Keywords:")
print(tfidf_df.head(20))
tfidf_df["keyword"].to_csv("full_output.csv", index=False)


Top TF-IDF Keywords:
        keyword     score
36         good  0.162052
5           app  0.112580
15         best  0.066466
58         nice  0.059026
11         bank  0.028694
89          use  0.021959
97      working  0.021785
37     good app  0.021566
16     best app  0.019837
46         like  0.019570
33         fast  0.018859
88       update  0.017628
31    excellent  0.017398
7   application  0.017079
12      banking  0.016898
77      service  0.016795
96         work  0.016719
59     nice app  0.016622
61           ok  0.016344
27         easy  0.014282


## Bank-Level Keyword Analysis

In [9]:
banks = df["bank"].unique()

for bank in banks:

    print(f"\n{'='*50}")
    print(f"Top Keywords for {bank}")
    print(f"{'='*50}")

    bank_reviews = df[
        df["bank"] == bank
    ]["clean_review"]

    vectorizer = TfidfVectorizer(
        stop_words='english',
        ngram_range=(1, 2),
        max_features=20
    )

    X_bank = vectorizer.fit_transform(bank_reviews)

    keywords_bank = vectorizer.get_feature_names_out()

    scores_bank = X_bank.mean(axis=0).A1

    bank_tfidf = pd.DataFrame({
        "keyword": keywords_bank,
        "score": scores_bank
    })

    bank_tfidf = bank_tfidf.sort_values(
        by="score",
        ascending=False
    )

    print(bank_tfidf)



Top Keywords for Commercial Bank of Ethiopia
        keyword     score
8          good  0.219316
0           app  0.140130
3          best  0.070014
11         nice  0.068565
4           cbe  0.039298
13           ok  0.037500
16       update  0.030889
2          bank  0.029433
19      working  0.029381
1   application  0.028862
14      service  0.028691
17          use  0.027250
9          like  0.027093
7          fast  0.024233
5          easy  0.023984
6     excellent  0.022864
15     transfer  0.020771
12     nice app  0.019972
18         work  0.019228
10       mobile  0.017936

Top Keywords for Bank of Abyssinia
           keyword     score
0              app  0.178401
8             good  0.167569
3             best  0.066030
11            nice  0.050263
1             bank  0.045934
4              boa  0.038362
18         working  0.035316
2          banking  0.031378
9           mobile  0.028404
6             fast  0.027019
17            work  0.025681
16             use  0.02

## Group Keywords into Themes

| Theme                                       | Related Keywords                                                                                                      | Business Meaning                                                                                   |
| ------------------------------------------- | --------------------------------------------------------------------------------------------------------------------- | -------------------------------------------------------------------------------------------------- |
| **User Satisfaction & Positive Experience** | good, best, nice, excellent, amazing, wow, love, happy, great, good job, app good, easy, easy use, secure, secured    | Users expressing satisfaction with the banking applications, usability, and service quality.       |
| **App Performance & Technical Issues**      | slow, doesnt work, error, keeps, poor, issue, problem, bad, worst, open, fix, tried, frequently, version, android     | Technical problems affecting reliability, speed, and app stability.                                |
| **Transaction & Banking Services**          | transfer, transaction, transactions, payment, money, balance, receipt, send, banking, mobile banking, amole, telebirr | Feedback related to transfers, payments, transaction processing, and digital banking services.     |
| **Account Access & Security**               | login, password, access, account, security, secured, phone, number                                                    | Issues or concerns related to authentication, account access, and security management.             |
| **Feature Improvement & User Requests**     | update, improve, need, needs, option, change, better, new, mobile app, application                                    | User suggestions and requests for application enhancements and new features.                       |
| **Bank & Service Reputation**               | cbe, boa, dashen, dashen bank, bank, banks, customer, service, use app                                                | General opinions and experiences associated with specific banks and their mobile banking services. |


In [20]:
num_topics = 5

nmf_model = NMF(
    n_components=num_topics,
    random_state=42
)

nmf_model.fit(X)

,"n_components n_components: int or {'auto'} or None, default='auto'Number of components. If `None`, all features are kept.If `n_components='auto'`, the number of components is automatically inferredfrom W or H shapes... versionchanged:: 1.4 Added `'auto'` value... versionchanged:: 1.6 Default value changed from `None` to `'auto'`.",5
,"init init: {'random', 'nndsvd', 'nndsvda', 'nndsvdar', 'custom'}, default=NoneMethod used to initialize the procedure.Valid options:- `None`: 'nndsvda' if n_components <= min(n_samples, n_features), otherwise random.- `'random'`: non-negative random matrices, scaled with: `sqrt(X.mean() / n_components)`- `'nndsvd'`: Nonnegative Double Singular Value Decomposition (NNDSVD) initialization (better for sparseness)- `'nndsvda'`: NNDSVD with zeros filled with the average of X (better when sparsity is not desired)- `'nndsvdar'` NNDSVD with zeros filled with small random values (generally faster, less accurate alternative to NNDSVDa for when sparsity is not desired)- `'custom'`: Use custom matrices `W` and `H` which must both be provided... versionchanged:: 1.1 When `init=None` and n_components is less than n_samples and n_features defaults to `nndsvda` instead of `nndsvd`.",None
,"solver solver: {'cd', 'mu'}, default='cd'Numerical solver to use:- 'cd' is a Coordinate Descent solver.- 'mu' is a Multiplicative Update solver... versionadded:: 0.17 Coordinate Descent solver... versionadded:: 0.19 Multiplicative Update solver.",'cd'
,"beta_loss beta_loss: float or {'frobenius', 'kullback-leibler', 'itakura-saito'}, default='frobenius'Beta divergence to be minimized, measuring the distance between Xand the dot product WH. Note that values different from 'frobenius'(or 2) and 'kullback-leibler' (or 1) lead to significantly slowerfits. Note that for beta_loss <= 0 (or 'itakura-saito'), the inputmatrix X cannot contain zeros. Used only in 'mu' solver... versionadded:: 0.19",'frobenius'
,"tol tol: float, default=1e-4Tolerance of the stopping condition.",0.0001
,"max_iter max_iter: int, default=200Maximum number of iterations before timing out.",200
,"random_state random_state: int, RandomState instance or None, default=NoneUsed for initialisation (when ``init`` == 'nndsvdar' or'random'), and in Coordinate Descent. Pass an int for reproducibleresults across multiple function calls.See :term:`Glossary `.",42
,"alpha_W alpha_W: float, default=0.0Constant that multiplies the regularization terms of `W`. Set it to zero(default) to have no regularization on `W`... versionadded:: 1.0",0.0
,"alpha_H alpha_H: float or ""same"", default=""same""Constant that multiplies the regularization terms of `H`. Set it to zero tohave no regularization on `H`. If ""same"" (default), it takes the same value as`alpha_W`... versionadded:: 1.0",'same'
,"l1_ratio l1_ratio: float, default=0.0The regularization mixing parameter, with 0 <= l1_ratio <= 1.For l1_ratio = 0 the penalty is an elementwise L2 penalty(aka Frobenius Norm).For l1_ratio = 1 it is an elementwise L1 penalty.For 0 < l1_ratio < 1, the penalty is a combination of L1 and L2... versionadded:: 0.17 Regularization parameter *l1_ratio* used in the Coordinate Descent solver.",0.0
,"verbose verbose: int, default=0Whether to be verbose.",0


In [24]:
feature_names = vectorizer.get_feature_names_out()

for topic_idx, topic in enumerate(nmf_model.components_):

    print(f"\nTopic {topic_idx + 1}")

    top_keywords = [
        feature_names[i]
        for i in topic.argsort()[:-11:-1]
    ]

    print(top_keywords)


Topic 1
['good', 'good app', 'service', 'application', 'good job', 'job', 'app good', 'update', 'work', 'use']

Topic 2
['app', 'good app', 'best app', 'nice app', 'time', 'great', 'amazing', 'like', 'app good', 'worst']

Topic 3
['best', 'best app', 'ethiopia', 'bank', 'used', 'experience', 'password', 'boa', 'apps', 'secured']

Topic 4
['nice', 'nice app', 'application', 'fast', 'use', 'easy', 'problem', 'wow', 'service', 'try']

Topic 5
['bank', 'working', 'use', 'banking', 'mobile', 'update', 'mobile banking', 'dashen', 'excellent', 'work']


| Theme                                   | Related Keywords                                                                    | Business Meaning                                                                                                                   |
| --------------------------------------- | ----------------------------------------------------------------------------------- | ---------------------------------------------------------------------------------------------------------------------------------- |
| **Service Quality & User Satisfaction** | good, good app, service, application, good job, job, app good, work, use, excellent | Users expressing positive experiences, satisfaction with banking services, and appreciation for app usability and reliability.     |
| **User Experience & App Perception**    | app, best app, nice app, amazing, great, like, app good, worst, time                | Customer opinions regarding overall app quality, usability, and user experience, including both positive and negative perceptions. |
| **Security & Banking Trust**            | password, secured, bank, experience, boa, ethiopia, apps                            | Feedback related to account security, trust in banking services, and user confidence in mobile banking platforms.                  |
| **App Performance & Ease of Use**       | nice, fast, easy, problem, service, try, use, application                           | Discussions about app speed, ease of navigation, convenience, and occasional usability or technical issues.                        |
| **Mobile Banking Functionality**        | bank, banking, mobile banking, mobile, working, update, dashen, work                | Reviews related to core banking features, mobile banking functionality, updates, and operational performance of the applications.  |
